In [26]:
import requests
import pandas as pd
import time

In [27]:
base = "https://api.openf1.org/v1"

sessions = requests.get(
    f"{base}/sessions",
    params={
        "year": 2025,
        "country_name": "Bahrain",
        "session_name": "Race"
    },
    timeout=30
)
sessions.raise_for_status()

session_key = sessions.json()[0]["session_key"]


## Functions for Data Manipulation

In [28]:
def fetch_data(endpoint):
    response = requests.get(
        f"{base}/{endpoint}",
        params={"session_key": session_key},
        timeout=120
    )
    response.raise_for_status()
    time.sleep(1)  
    return pd.DataFrame(response.json())

In [29]:
def merge_tables(lap_table,stint_table)->pd.DataFrame:
    merged_table=pd.merge(lap_table,stint_table,on=["driver_number","session_key","meeting_key"],how="left")
    merged_table=merged_table[merged_table["lap_number"].between(merged_table["lap_start"], merged_table["lap_end"])].copy()
    merged_table["stint_lap"]=(merged_table["lap_number"]-merged_table["lap_start"]+1)
    merged_table["tyre_age"]=(merged_table["tyre_age_at_start"]+merged_table["stint_lap"]-1)
    return merged_table

In [30]:

stints = fetch_data("stints")
weather = fetch_data("weather")
pit_stops = fetch_data("pit")
laps=fetch_data("laps")
stints.to_csv("bahrain_2025_stints.csv", index=False)
weather.to_csv("bahrain_2025_weather.csv", index=False)
pit_stops.to_csv("bahrain_2025_pit_stops.csv", index=False)
laps.to_csv("bahrain_2025_laps.csv", index=False)

for name, df in [
    ("Laps", laps),
    ("Stints", stints),
    ("Weather", weather),
    ("Pit Stops", pit_stops)
]:
    print(f"\n{name}: {df.shape}")
    print(df.columns.tolist())
    print(df.head(3).to_string(index=False))


Laps: (1129, 16)
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed']
 meeting_key  session_key  driver_number  lap_number                       date_start  duration_sector_1  duration_sector_2  duration_sector_3  i1_speed  i2_speed  is_pit_out_lap  lap_duration                                         segments_sector_1                                                  segments_sector_2                          segments_sector_3  st_speed
        1257        10014             81           1 2025-04-13T15:03:49.173000+00:00             33.127             42.130             23.956     233.0     244.0           False        99.213 [None, 0, 2049, 2049, 2051, 2051, 2049, 2049, 2051, 2049] [2051, 2051, 2051, 2049, 2051, 2049, 2049, 2049, 2051, 2049, 2049] [2049, 2049, 2049, 20

In [31]:
merged=merge_tables(laps, stints)
merged.to_csv("bahrain_2025_merged_laps_stints.csv", index=False)

print("Original laps:", len(laps))
print("Merged laps:", len(merged))

print(
    "Duplicate laps:",
    merged.duplicated(
        subset=["session_key", "driver_number", "lap_number"]
    ).sum()
)

print(merged[[
    "driver_number",
    "lap_number",
    "stint_number",
    "compound",
    "stint_lap",
    "tyre_age",
    "lap_duration"
]].head(15).to_string(index=False))

Original laps: 1129
Merged laps: 1128
Duplicate laps: 0
 driver_number  lap_number  stint_number compound  stint_lap  tyre_age  lap_duration
            81           1             1     SOFT          1         3        99.213
            16           1             1   MEDIUM          1         0       101.765
            63           1             1     SOFT          1         3        99.946
            55           1             1     SOFT          1         3       102.998
            10           1             1     SOFT          1         3       102.327
            12           1             1     SOFT          1         3       103.695
            22           1             1     SOFT          1         3       105.323
             7           1             1     SOFT          1         0       105.518
             6           1             1     SOFT          1         0       108.674
             4           1             1     SOFT          1         3       100.891
         

In [32]:
def merge_weather(lap_table, weather_table) -> pd.DataFrame:
    laps = lap_table.copy()

    weather = weather_table[[
        "session_key",
        "date",
        "track_temperature",
        "air_temperature",
        "humidity",
        "rainfall"
    ]].copy()

    laps["date_start"] = pd.to_datetime(
        laps["date_start"], utc=True
    )
    weather["date"] = pd.to_datetime(
        weather["date"], utc=True
    )

    laps = laps.sort_values("date_start")
    weather = weather.sort_values("date")

    merged = pd.merge_asof(
        laps,
        weather,
        left_on="date_start",
        right_on="date",
        by="session_key",
        direction="backward",
        tolerance=pd.Timedelta("10min")
    )

    return merged

In [33]:
final_data = merge_weather(merged, weather)

print("Total rows:", len(final_data))
print("Missing weather:", final_data["track_temperature"].isna().sum())

print(final_data[[
    "driver_number",
    "lap_number",
    "compound",
    "tyre_age",
    "lap_duration",
    "track_temperature",
    "air_temperature",
    "humidity",
    "rainfall"
]].head(15).to_string(index=False))
final_data.to_csv("bahrain_2025_final_data.csv", index=False)

Total rows: 1128
Missing weather: 0
 driver_number  lap_number compound  tyre_age  lap_duration  track_temperature  air_temperature  humidity  rainfall
            81           1     SOFT         3        99.213               32.2             27.2      44.0         0
            87           1     SOFT         0       107.250               32.2             27.2      44.0         0
            18           1     SOFT         0       109.164               32.2             27.2      44.0         0
             5           1   MEDIUM         0       109.810               32.2             27.2      44.0         0
            30           1   MEDIUM         0       109.059               32.2             27.2      44.0         0
            44           1   MEDIUM         0       105.026               32.2             27.2      44.0         0
            23           1     SOFT         0       107.051               32.2             27.2      44.0         0
            14           1   MEDIUM 

/tmp/ipykernel_16891/861842173.py:30: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  tolerance=pd.Timedelta("10min")
